# V-JEPA 2 for Active Matter Physical Simulation

**NYU Deep Learning Spring 2026 — Final Project**

## Overview
This notebook implements a V-JEPA 2-style representation learning pipeline for the
`polymathic-ai/active_matter` dataset from [The Well](https://huggingface.co/polymathic-ai).

**Dataset:** Each sample is a simulation trajectory of 16 frames × 224×224 × 11 channels,
governed by two physical parameters α (active dipole strength) and ζ (steric alignment).

**Method:** V-JEPA 2 (Joint-Embedding Predictive Architecture for Video)
- Context encoder (ViT) encodes visible (unmasked) 3D patches
- Target encoder (EMA copy) encodes all patches — provides stable targets
- Predictor (narrow ViT) predicts target representations from context
- No labels used during pre-training

**Evaluation:** Linear probing + kNN regression to predict α and ζ (MSE on z-scored labels)

**Constraints:**
- < 100M parameters total
- No pretrained weights
- Only `active_matter` data
- No fine-tuning for evaluation — frozen backbone only


## 1. Install Dependencies


In [ ]:
# Run once to install required packages
# !pip install torch torchvision datasets huggingface_hub
# !pip install scikit-learn wandb matplotlib h5py
# !pip install the_well  # optional: official The Well package


## 2. Imports


In [ ]:
import os
import math
import copy
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from datasets import load_dataset

from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False
    print('wandb not found — experiment tracking disabled')

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 3. Configuration


In [ ]:
import os

NETID = os.environ.get('NETID', 'kk5991')  # set your NYU NetID

@dataclass
class Config:
    # ── Data ──────────────────────────────────────────────────────────────
    dataset_name: str          = 'polymathic-ai/active_matter'
    data_dir: str              = f'/scratch/{NETID}/data/active_matter'
    num_frames: int            = 16    # frames per clip
    img_size: int              = 224   # spatial resolution (H = W)
    num_channels: int          = 11   # 1 concentration + 2 velocity + 4 orient + 4 strain-rate

    # ── Patching ──────────────────────────────────────────────────────────
    # 3D tubelet: (tubelet_size, spatial_patch_size, spatial_patch_size)
    spatial_patch_size: int    = 16   # 224/16 = 14 patches per spatial dim
    tubelet_size: int          = 2    # 16/2  =  8 temporal patches
    # Total patches = 8 * 14 * 14 = 1568

    # ── Encoder (ViT-Small) ───────────────────────────────────────────────
    # ~2.5M params — well within the 100M limit
    embed_dim: int             = 384
    encoder_depth: int         = 12
    encoder_num_heads: int     = 6
    mlp_ratio: float           = 4.0

    # ── Predictor (narrow ViT) ────────────────────────────────────────────
    predictor_embed_dim: int   = 192
    predictor_depth: int       = 6
    predictor_num_heads: int   = 6

    # ── Training ──────────────────────────────────────────────────────────
    batch_size: int            = 4     # per-GPU; adjust based on VRAM
    grad_accum_steps: int      = 2     # effective batch = 8
    lr: float                  = 1.5e-4
    min_lr: float              = 1e-6
    weight_decay: float        = 0.05
    epochs: int                = 100
    warmup_epochs: int         = 10
    clip_grad: float           = 1.0

    # ── EMA ───────────────────────────────────────────────────────────────
    ema_decay_start: float     = 0.996
    ema_decay_end: float       = 1.0

    # ── Masking (V-JEPA multi-block) ──────────────────────────────────────
    context_mask_scale: Tuple[float, float] = (0.85, 1.0)   # encoder sees ~85-100%
    target_mask_scale: Tuple[float, float]  = (0.15, 0.2)   # predict ~15-20% blocks
    num_target_blocks: int     = 4
    aspect_ratio: Tuple[float, float]       = (0.75, 1.5)

    # ── Checkpoint ────────────────────────────────────────────────────────
    checkpoint_dir: str        = f'/scratch/{NETID}/checkpoints/vjepa2'
    log_interval: int          = 50
    save_interval: int         = 5

    # ── Evaluation ────────────────────────────────────────────────────────
    eval_interval: int         = 10   # evaluate every N epochs
    knn_k: int                 = 20

    # ── Misc ──────────────────────────────────────────────────────────────
    seed: int                  = 42
    device: str                = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers: int           = 4
    use_wandb: bool            = True
    wandb_project: str         = 'vjepa2-active-matter'

cfg = Config()

num_t = cfg.num_frames // cfg.tubelet_size
num_h = cfg.img_size   // cfg.spatial_patch_size
num_w = cfg.img_size   // cfg.spatial_patch_size
num_patches = num_t * num_h * num_w

print(f'Device          : {cfg.device}')
print(f'Temporal patches: {num_t}')
print(f'Spatial patches : {num_h} x {num_w}')
print(f'Total patches   : {num_patches}')


## 4. Reproducibility


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Uncomment for full determinism (may slow training):
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

set_seed(cfg.seed)
print(f'Seed set to {cfg.seed}')


## 5. Dataset


In [ ]:
class ActiveMatterDataset(Dataset):
    """
    Dataset wrapper for polymathic-ai/active_matter.

    Each trajectory sample:
      - frames : Tensor (T, C, H, W) float32  — simulation fields
      - alpha  : float — active dipole strength (5 discrete values; NOT used in training)
      - zeta   : float — steric alignment       (9 discrete values; NOT used in training)

    The physical parameters are attached only for the evaluation stage.
    """

    # Map from The Well HuggingFace field names to our canonical names.
    # Adjust if the actual schema differs.
    FIELD_KEYS  = ['data', 'fields', 'simulation_data']      # try in order
    ALPHA_KEYS  = ['alpha', 'params/alpha', 'active_dipole']
    ZETA_KEYS   = ['zeta',  'params/zeta',  'steric_alignment']

    def __init__(
        self,
        split: str           = 'train',
        data_dir: Optional[str] = None,
        num_frames: int      = 16,
        img_size: int        = 224,
        num_channels: int    = 11,
        normalize: bool      = True,
        streaming: bool      = False,
    ):
        self.split       = split
        self.num_frames  = num_frames
        self.img_size    = img_size
        self.num_channels = num_channels
        self.normalize   = normalize

        print(f'Loading {split} split from HuggingFace ...')
        self.dataset = load_dataset(
            'polymathic-ai/active_matter',
            split=split,
            cache_dir=data_dir,
            trust_remote_code=True,
            streaming=streaming,
        )
        if not streaming:
            print(f'  {len(self.dataset):,} samples')

    def __len__(self):
        return len(self.dataset)

    @staticmethod
    def _get_field(sample: dict, keys: List[str], default=0.0):
        """Try multiple key names to handle schema variations."""
        for k in keys:
            if k in sample:
                return sample[k]
        return default

    def __getitem__(self, idx: int) -> dict:
        sample = self.dataset[idx]

        # ── Extract fields array ───────────────────────────────────────
        data = self._get_field(sample, self.FIELD_KEYS)
        data = np.array(data, dtype=np.float32)   # expected: (T, H, W, C)

        # ── Temporal subsampling ───────────────────────────────────────
        T = data.shape[0]
        if T > self.num_frames:
            start = random.randint(0, T - self.num_frames)
            data  = data[start : start + self.num_frames]
        elif T < self.num_frames:
            # Repeat last frame to pad
            pad = np.repeat(data[[-1]], self.num_frames - T, axis=0)
            data = np.concatenate([data, pad], axis=0)

        # ── Spatial crop ───────────────────────────────────────────────
        H, W = data.shape[1], data.shape[2]
        if H > self.img_size:
            h0 = random.randint(0, H - self.img_size)
            data = data[:, h0 : h0 + self.img_size, :, :]
        if W > self.img_size:
            w0 = random.randint(0, W - self.img_size)
            data = data[:, :, w0 : w0 + self.img_size, :]

        # ── (T, H, W, C) → (T, C, H, W) ──────────────────────────────
        data = torch.from_numpy(data).permute(0, 3, 1, 2)   # (T, C, H, W)

        # ── Per-channel z-score normalisation ─────────────────────────
        if self.normalize:
            mean = data.mean(dim=(0, 2, 3), keepdim=True)
            std  = data.std( dim=(0, 2, 3), keepdim=True).clamp(min=1e-6)
            data = (data - mean) / std

        # ── Physical parameters (evaluation only) ──────────────────────
        alpha = float(self._get_field(sample, self.ALPHA_KEYS, default=0.0))
        zeta  = float(self._get_field(sample, self.ZETA_KEYS,  default=0.0))

        return {
            'frames': data,                                  # (T, C, H, W)
            'alpha' : torch.tensor(alpha, dtype=torch.float32),
            'zeta'  : torch.tensor(zeta,  dtype=torch.float32),
        }


def build_dataloaders(cfg: Config):
    train_ds = ActiveMatterDataset(
        'train', data_dir=cfg.data_dir,
        num_frames=cfg.num_frames, img_size=cfg.img_size, num_channels=cfg.num_channels,
    )
    val_ds = ActiveMatterDataset(
        'validation', data_dir=cfg.data_dir,
        num_frames=cfg.num_frames, img_size=cfg.img_size, num_channels=cfg.num_channels,
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
    )
    return train_loader, val_loader

print('Dataset class defined.')
print('(Do not load data here — run build_dataloaders() in main())')


## 6. 3D Patch Embedding


In [ ]:
class PatchEmbed3D(nn.Module):
    """
    Embed a video clip (B, T, C, H, W) into a flat sequence of patch tokens
    using a single 3D convolution with stride = kernel = tubelet.

    Output: (B, num_patches, embed_dim)
    """

    def __init__(
        self,
        num_frames: int        = 16,
        img_size: int          = 224,
        num_channels: int      = 11,
        spatial_patch_size: int = 16,
        tubelet_size: int      = 2,
        embed_dim: int         = 384,
    ):
        super().__init__()
        self.num_t = num_frames // tubelet_size
        self.num_h = img_size   // spatial_patch_size
        self.num_w = img_size   // spatial_patch_size
        self.num_patches = self.num_t * self.num_h * self.num_w

        # 3D conv: (B, C, T, H, W) -> (B, embed_dim, num_t, num_h, num_w)
        self.proj = nn.Conv3d(
            in_channels  = num_channels,
            out_channels = embed_dim,
            kernel_size  = (tubelet_size, spatial_patch_size, spatial_patch_size),
            stride       = (tubelet_size, spatial_patch_size, spatial_patch_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, T, C, H, W)
        -> (B, num_patches, embed_dim)
        """
        B, T, C, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4)          # (B, C, T, H, W)
        x = self.proj(x)                       # (B, E, num_t, num_h, num_w)
        x = x.flatten(2).transpose(1, 2)       # (B, num_patches, E)
        return x


# Quick shape check
_dummy = PatchEmbed3D(
    num_frames=cfg.num_frames, img_size=cfg.img_size, num_channels=cfg.num_channels,
    spatial_patch_size=cfg.spatial_patch_size, tubelet_size=cfg.tubelet_size, embed_dim=cfg.embed_dim,
)
_x = torch.randn(2, cfg.num_frames, cfg.num_channels, cfg.img_size, cfg.img_size)
_out = _dummy(_x)
print(f'PatchEmbed3D output: {_out.shape}  (expected: [2, {num_patches}, {cfg.embed_dim}])')
del _dummy, _x, _out


## 7. ViT Building Blocks


In [ ]:
class Attention(nn.Module):
    def __init__(
        self, dim: int, num_heads: int = 8,
        qkv_bias: bool = True, attn_drop: float = 0., proj_drop: float = 0.,
    ):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj      = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        qkv = (
            self.qkv(x)
                .reshape(B, N, 3, self.num_heads, C // self.num_heads)
                .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv.unbind(0)          # each: (B, heads, N, head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int, drop: float = 0.):
        super().__init__()
        self.fc1  = nn.Linear(in_features,  hidden_features)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(
        self, dim: int, num_heads: int, mlp_ratio: float = 4.,
        drop: float = 0., attn_drop: float = 0.,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = Attention(dim, num_heads=num_heads, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, int(dim * mlp_ratio), dim, drop=drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


print('ViT building blocks defined.')


## 8. 3D Sinusoidal Positional Encoding


In [ ]:
def get_3d_sincos_pos_embed(embed_dim: int, num_t: int, num_h: int, num_w: int) -> torch.Tensor:
    """
    Build factored 3D sinusoidal positional embeddings.
    Concatenates independent 1D encodings for T, H, W.

    embed_dim must be divisible by 3 (one third per axis).
    Returns: (num_t * num_h * num_w, embed_dim)
    """
    assert embed_dim % 3 == 0, f'embed_dim must be divisible by 3, got {embed_dim}'
    d = embed_dim // 3

    def sincos_1d(length: int, dim: int) -> torch.Tensor:
        pos = torch.arange(length, dtype=torch.float32).unsqueeze(1)     # (L, 1)
        i   = torch.arange(0, dim, 2, dtype=torch.float32)               # (dim/2,)
        div = torch.exp(-i * math.log(10_000.0) / dim)
        pe  = torch.zeros(length, dim)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe  # (L, dim)

    pe_t = sincos_1d(num_t, d)  # (num_t, d)
    pe_h = sincos_1d(num_h, d)  # (num_h, d)
    pe_w = sincos_1d(num_w, d)  # (num_w, d)

    # Broadcast over the 3 axes, then concatenate along feature dim
    pe_t = pe_t[:, None, None, :].expand(num_t, num_h, num_w, d)
    pe_h = pe_h[None, :, None, :].expand(num_t, num_h, num_w, d)
    pe_w = pe_w[None, None, :, :].expand(num_t, num_h, num_w, d)

    pe = torch.cat([pe_t, pe_h, pe_w], dim=-1)  # (num_t, num_h, num_w, embed_dim)
    return pe.reshape(-1, embed_dim)              # (num_patches, embed_dim)


_pe = get_3d_sincos_pos_embed(cfg.embed_dim, num_t, num_h, num_w)
print(f'Positional embedding shape: {_pe.shape}  (expected: [{num_patches}, {cfg.embed_dim}])')
del _pe


## 9. Context / Target Encoder


In [ ]:
class VJEPAEncoder(nn.Module):
    """
    ViT-based encoder used as both context encoder and (EMA copy) target encoder.

    When used as context encoder: receives only the *unmasked* patch subset.
    When used as target encoder : receives all patches (keep_ids=None).
    """

    def __init__(
        self,
        num_frames: int         = 16,
        img_size: int           = 224,
        num_channels: int       = 11,
        spatial_patch_size: int = 16,
        tubelet_size: int       = 2,
        embed_dim: int          = 384,
        depth: int              = 12,
        num_heads: int          = 6,
        mlp_ratio: float        = 4.,
        drop_rate: float        = 0.,
        attn_drop_rate: float   = 0.,
    ):
        super().__init__()
        self.embed_dim = embed_dim

        self.patch_embed = PatchEmbed3D(
            num_frames=num_frames, img_size=img_size, num_channels=num_channels,
            spatial_patch_size=spatial_patch_size, tubelet_size=tubelet_size,
            embed_dim=embed_dim,
        )
        _num_t = num_frames // tubelet_size
        _num_h = img_size   // spatial_patch_size
        _num_w = img_size   // spatial_patch_size

        # Fixed (non-learned) positional embedding — registered as buffer
        pos_emb = get_3d_sincos_pos_embed(embed_dim, _num_t, _num_h, _num_w)  # (N, E)
        self.register_buffer('pos_embed', pos_emb.unsqueeze(0))               # (1, N, E)

        self.blocks = nn.Sequential(*[
            Block(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(
        self,
        x: torch.Tensor,
        keep_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        x        : (B, T, C, H, W)
        keep_ids : (B, num_keep) — patch indices to keep (context masking)
                   Pass None to encode all patches (target encoder usage).
        Returns  : (B, num_keep | N, embed_dim)
        """
        tokens = self.patch_embed(x)          # (B, N, E)
        tokens = tokens + self.pos_embed      # add positional encoding

        if keep_ids is not None:
            E = tokens.shape[-1]
            idx = keep_ids.unsqueeze(-1).expand(-1, -1, E)
            tokens = torch.gather(tokens, dim=1, index=idx)

        tokens = self.blocks(tokens)
        tokens = self.norm(tokens)
        return tokens


print('VJEPAEncoder defined.')


## 10. Predictor


In [ ]:
class VJEPAPredictor(nn.Module):
    """
    Narrow transformer that predicts target patch representations
    from context tokens.

    Pipeline:
      1. Project context: encoder_dim -> predictor_dim
      2. Add positional embeddings to context tokens
      3. Build mask tokens for target positions (learnable + positional embed)
      4. Concatenate context + mask tokens, run transformer
      5. Extract target outputs, project back to encoder_dim
    """

    def __init__(
        self,
        num_patches: int,
        encoder_embed_dim: int    = 384,
        predictor_embed_dim: int  = 192,
        depth: int                = 6,
        num_heads: int            = 6,
        mlp_ratio: float          = 4.,
    ):
        super().__init__()
        self.pred_dim = predictor_embed_dim

        # Positional embedding for ALL patch positions (in predictor space)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, predictor_embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Learnable mask token (position-agnostic part)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_embed_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        self.encoder_to_predictor  = nn.Linear(encoder_embed_dim,    predictor_embed_dim)
        self.predictor_to_encoder  = nn.Linear(predictor_embed_dim,  encoder_embed_dim)

        self.blocks = nn.Sequential(*[
            Block(predictor_embed_dim, num_heads, mlp_ratio)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(predictor_embed_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self,
        context_tokens: torch.Tensor,   # (B, num_ctx, encoder_dim)
        context_ids: torch.Tensor,      # (B, num_ctx)
        target_ids: torch.Tensor,       # (B, num_tgt)
    ) -> torch.Tensor:
        """Returns: (B, num_tgt, encoder_dim)"""
        B, num_ctx, _ = context_tokens.shape
        num_tgt = target_ids.shape[1]
        D = self.pred_dim

        # 1. Project context
        ctx = self.encoder_to_predictor(context_tokens)          # (B, num_ctx, D)

        # 2. Add positional embedding to context tokens
        all_pos = self.pos_embed.expand(B, -1, -1)               # (B, N, D)
        ctx_pos = torch.gather(all_pos, 1, context_ids.unsqueeze(-1).expand(-1, -1, D))
        ctx = ctx + ctx_pos

        # 3. Build mask tokens for target positions
        tgt_pos  = torch.gather(all_pos, 1, target_ids.unsqueeze(-1).expand(-1, -1, D))
        mask_tok = self.mask_token.expand(B, num_tgt, -1) + tgt_pos

        # 4. Run predictor
        tokens = torch.cat([ctx, mask_tok], dim=1)               # (B, num_ctx+num_tgt, D)
        tokens = self.blocks(tokens)
        tokens = self.norm(tokens)

        # 5. Extract target outputs and project to encoder dim
        preds = tokens[:, num_ctx:, :]                           # (B, num_tgt, D)
        return self.predictor_to_encoder(preds)                  # (B, num_tgt, encoder_dim)


print('VJEPAPredictor defined.')


## 11. V-JEPA 2 Full Model


In [ ]:
class VJEPA2(nn.Module):
    """
    V-JEPA 2 — Joint-Embedding Predictive Architecture for Video.

    Components:
      context_encoder : encodes the visible (unmasked) patches
      target_encoder  : EMA copy of context encoder; encodes ALL patches
      predictor       : maps context representations to target representations

    Only context_encoder and predictor are updated via gradient descent.
    target_encoder is updated via EMA — never receives gradients.
    """

    def __init__(self, cfg: Config):
        super().__init__()

        enc_kwargs = dict(
            num_frames=cfg.num_frames, img_size=cfg.img_size, num_channels=cfg.num_channels,
            spatial_patch_size=cfg.spatial_patch_size, tubelet_size=cfg.tubelet_size,
            embed_dim=cfg.embed_dim, depth=cfg.encoder_depth, num_heads=cfg.encoder_num_heads,
            mlp_ratio=cfg.mlp_ratio,
        )
        self.context_encoder = VJEPAEncoder(**enc_kwargs)
        self.target_encoder  = copy.deepcopy(self.context_encoder)

        # Target encoder is never updated via backprop
        for p in self.target_encoder.parameters():
            p.requires_grad_(False)

        num_patches = self.context_encoder.patch_embed.num_patches
        self.predictor = VJEPAPredictor(
            num_patches=num_patches,
            encoder_embed_dim=cfg.embed_dim,
            predictor_embed_dim=cfg.predictor_embed_dim,
            depth=cfg.predictor_depth,
            num_heads=cfg.predictor_num_heads,
            mlp_ratio=cfg.mlp_ratio,
        )

    @torch.no_grad()
    def update_target_encoder(self, decay: float):
        """EMA update: target <- decay * target + (1 - decay) * context."""
        for p_ctx, p_tgt in zip(
            self.context_encoder.parameters(),
            self.target_encoder.parameters(),
        ):
            p_tgt.data.mul_(decay).add_(p_ctx.data, alpha=1.0 - decay)

    def forward(
        self,
        x: torch.Tensor,
        context_ids: torch.Tensor,
        target_ids: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x           : (B, T, C, H, W)
        context_ids : (B, num_ctx)  — patch indices visible to context encoder
        target_ids  : (B, num_tgt) — patch indices to predict

        Returns:
          preds   : (B, num_tgt, embed_dim) — predictor output
          targets : (B, num_tgt, embed_dim) — target encoder output (stop-grad)
        """
        # Context encoder: masked input
        context_tokens = self.context_encoder(x, keep_ids=context_ids)

        # Target encoder: full input, gather target positions
        with torch.no_grad():
            all_tokens = self.target_encoder(x, keep_ids=None)   # (B, N, E)
            E = all_tokens.shape[-1]
            idx = target_ids.unsqueeze(-1).expand(-1, -1, E)
            targets = torch.gather(all_tokens, 1, idx)           # (B, num_tgt, E)

        preds = self.predictor(context_tokens, context_ids, target_ids)
        return preds, targets


def count_parameters(model: nn.Module) -> int:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total parameters    : {total:>12,}')
    print(f'  Trainable parameters: {trainable:>12,}')
    assert total < 100_000_000, f'Model exceeds 100M parameter limit! ({total:,})'
    return total


# Instantiate and count params
_model = VJEPA2(cfg)
print('Parameter count:')
count_parameters(_model)
del _model


## 12. Multi-Block 3D Masking


In [ ]:
def _sample_block(
    num_t: int, num_h: int, num_w: int,
    scale: Tuple[float, float],
    aspect_ratio: Tuple[float, float],
    device: torch.device,
) -> torch.Tensor:
    """
    Sample one 3D block mask.
    Returns: bool tensor (num_t * num_h * num_w,), True = masked.
    """
    N = num_t * num_h * num_w
    target_area = random.uniform(*scale) * N

    for _ in range(10):                  # retry loop for valid aspect ratio
        ar = random.uniform(*aspect_ratio)
        bh = max(1, int(round(math.sqrt(target_area / num_t * ar))))
        bw = max(1, int(round(math.sqrt(target_area / num_t / ar))))
        bt = max(1, int(round(target_area / (bh * bw))))
        if bh <= num_h and bw <= num_w and bt <= num_t:
            break
    bh = min(bh, num_h)
    bw = min(bw, num_w)
    bt = min(bt, num_t)

    t0 = random.randint(0, num_t - bt)
    h0 = random.randint(0, num_h - bh)
    w0 = random.randint(0, num_w - bw)

    mask = torch.zeros(num_t, num_h, num_w, dtype=torch.bool, device=device)
    mask[t0:t0+bt, h0:h0+bh, w0:w0+bw] = True
    return mask.reshape(-1)


def build_masks(
    batch_size: int,
    num_t: int, num_h: int, num_w: int,
    context_scale: Tuple[float, float],
    target_scale: Tuple[float, float],
    num_target_blocks: int,
    aspect_ratio: Tuple[float, float],
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build batched context / target masks.

    Strategy (V-JEPA):
      * Sample num_target_blocks 3D blocks → union forms the target mask.
      * Context = complement of target mask.

    Returns:
      context_ids : (B, max_ctx) — indices of context patches (padded with 0)
      target_ids  : (B, max_tgt) — indices of target  patches (padded with 0)
    """
    ctx_list, tgt_list = [], []

    for _ in range(batch_size):
        # Target: union of multiple blocks
        tgt_mask = torch.zeros(num_t * num_h * num_w, dtype=torch.bool, device=device)
        for _ in range(num_target_blocks):
            tgt_mask |= _sample_block(num_t, num_h, num_w, target_scale, aspect_ratio, device)

        ctx_ids = torch.where(~tgt_mask)[0]
        tgt_ids = torch.where(tgt_mask)[0]
        ctx_list.append(ctx_ids)
        tgt_list.append(tgt_ids)

    max_ctx = max(c.shape[0] for c in ctx_list)
    max_tgt = max(t.shape[0] for t in tgt_list)

    # Pad with zeros (padding tokens are ignored in loss via length tracking,
    # or simply ignored since we take fixed-length slices in practice)
    context_ids = torch.stack([F.pad(c, (0, max_ctx - c.shape[0])) for c in ctx_list])
    target_ids  = torch.stack([F.pad(t, (0, max_tgt - t.shape[0])) for t in tgt_list])

    return context_ids, target_ids


# Quick test
_ctx, _tgt = build_masks(2, num_t, num_h, num_w,
                          cfg.context_mask_scale, cfg.target_mask_scale,
                          cfg.num_target_blocks, cfg.aspect_ratio, torch.device('cpu'))
print(f'context_ids shape: {_ctx.shape}')
print(f'target_ids  shape: {_tgt.shape}')
print(f'Context ratio: {_ctx.shape[1]/num_patches:.2%} | Target ratio: {_tgt.shape[1]/num_patches:.2%}')
del _ctx, _tgt


## 13. Loss and Learning Rate / EMA Schedules


In [ ]:
def vjepa_loss(preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Smooth L1 on L2-normalised representations.
    Normalisation discourages collapse while being robust to outliers.
    """
    preds   = F.normalize(preds,   dim=-1)
    targets = F.normalize(targets, dim=-1)
    return F.smooth_l1_loss(preds, targets.detach())


def cosine_decay(
    base_value: float, final_value: float, total_steps: int
) -> np.ndarray:
    steps = np.arange(total_steps)
    return final_value + 0.5 * (base_value - final_value) * (
        1 + np.cos(np.pi * steps / total_steps)
    )


def build_lr_schedule(cfg: Config, steps_per_epoch: int) -> np.ndarray:
    total_steps  = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    warmup = np.linspace(0.0, cfg.lr, warmup_steps)
    decay  = cosine_decay(cfg.lr, cfg.min_lr, total_steps - warmup_steps)
    return np.concatenate([warmup, decay])


def build_ema_schedule(cfg: Config, total_steps: int) -> np.ndarray:
    return cosine_decay(cfg.ema_decay_start, cfg.ema_decay_end, total_steps)


print('Loss and schedules defined.')


## 14. Training Loop


In [ ]:
def train_one_epoch(
    model: VJEPA2,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    lr_schedule: np.ndarray,
    ema_schedule: np.ndarray,
    global_step: int,
    epoch: int,
    cfg: Config,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
) -> Tuple[float, int]:
    """
    Run one full training epoch.

    Returns: (mean_loss, updated_global_step)
    """
    model.train()
    device  = torch.device(cfg.device)
    _num_t  = cfg.num_frames // cfg.tubelet_size
    _num_h  = cfg.img_size   // cfg.spatial_patch_size
    _num_w  = cfg.img_size   // cfg.spatial_patch_size
    use_amp = (cfg.device == 'cuda')

    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        # ── Apply LR and EMA schedules ─────────────────────────────
        lr_idx  = min(global_step, len(lr_schedule)  - 1)
        ema_idx = min(global_step, len(ema_schedule) - 1)
        for pg in optimizer.param_groups:
            pg['lr'] = float(lr_schedule[lr_idx])
        ema_decay = float(ema_schedule[ema_idx])

        frames = batch['frames'].to(device, non_blocking=True)   # (B, T, C, H, W)

        # ── Sample masks ───────────────────────────────────────────
        context_ids, target_ids = build_masks(
            batch_size        = frames.shape[0],
            num_t=_num_t, num_h=_num_h, num_w=_num_w,
            context_scale     = cfg.context_mask_scale,
            target_scale      = cfg.target_mask_scale,
            num_target_blocks = cfg.num_target_blocks,
            aspect_ratio      = cfg.aspect_ratio,
            device            = device,
        )

        # ── Forward + loss ─────────────────────────────────────────
        with torch.cuda.amp.autocast(enabled=use_amp):
            preds, targets = model(frames, context_ids, target_ids)
            loss = vjepa_loss(preds, targets) / cfg.grad_accum_steps

        # ── Backward ───────────────────────────────────────────────
        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # ── Gradient accumulation step ─────────────────────────────
        if (step + 1) % cfg.grad_accum_steps == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
                optimizer.step()
            optimizer.zero_grad()

            # EMA update of target encoder
            model.update_target_encoder(ema_decay)
            global_step += 1

        running_loss += loss.item() * cfg.grad_accum_steps

        if step % cfg.log_interval == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(
                f'  Epoch {epoch:3d} | Step {step:5d}/{len(loader)} '
                f'| Loss {loss.item() * cfg.grad_accum_steps:.4f} '
                f'| LR {current_lr:.2e} | EMA {ema_decay:.5f}'
            )

    return running_loss / len(loader), global_step


## 15. Checkpointing


In [ ]:
def save_checkpoint(
    model: VJEPA2,
    optimizer: optim.Optimizer,
    scaler,
    epoch: int,
    global_step: int,
    loss: float,
    cfg: Config,
) -> str:
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    path = os.path.join(cfg.checkpoint_dir, f'vjepa2_epoch{epoch:04d}.pt')
    torch.save({
        'epoch'             : epoch,
        'global_step'       : global_step,
        'model_state_dict'  : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict' : scaler.state_dict() if scaler else None,
        'loss'              : loss,
        'config'            : vars(cfg),
    }, path)
    print(f'  Checkpoint saved -> {path}')
    return path


def load_checkpoint(
    model: VJEPA2,
    optimizer: optim.Optimizer,
    scaler,
    path: str,
    cfg: Config,
) -> Tuple[int, int]:
    ckpt = torch.load(path, map_location=cfg.device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scaler and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    epoch       = ckpt['epoch']
    global_step = ckpt['global_step']
    print(f'  Resumed from epoch {epoch}, global_step {global_step}')
    return epoch, global_step


def find_latest_checkpoint(checkpoint_dir: str) -> Optional[str]:
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    ckpts = sorted(Path(checkpoint_dir).glob('vjepa2_epoch*.pt'))
    return str(ckpts[-1]) if ckpts else None


print('Checkpointing utilities defined.')


## 16. Feature Extraction (Frozen Encoder)


In [ ]:
@torch.no_grad()
def extract_features(
    model: VJEPA2,
    loader: DataLoader,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract global-average-pooled representations from the (frozen) target encoder.

    Returns:
      features : (N, embed_dim)
      alphas   : (N,)
      zetas    : (N,)
    """
    model.eval()
    device = torch.device(cfg.device)
    feats_list, alpha_list, zeta_list = [], [], []

    for batch in loader:
        frames = batch['frames'].to(device)                    # (B, T, C, H, W)
        # Use target encoder for stable, EMA-smoothed representations
        tokens = model.target_encoder(frames, keep_ids=None)  # (B, N, E)
        feats  = tokens.mean(dim=1)                           # (B, E) global avg pool
        feats_list.append(feats.cpu().float().numpy())
        alpha_list.append(batch['alpha'].numpy())
        zeta_list.append(batch['zeta'].numpy())

    return (
        np.concatenate(feats_list, axis=0),
        np.concatenate(alpha_list, axis=0),
        np.concatenate(zeta_list,  axis=0),
    )


print('Feature extraction function defined.')


## 17. Linear Probe Evaluation


In [ ]:
def linear_probe_eval(
    train_feats : np.ndarray, train_alpha : np.ndarray, train_zeta : np.ndarray,
    val_feats   : np.ndarray, val_alpha   : np.ndarray, val_zeta   : np.ndarray,
    ridge_alpha : float = 1.0,
) -> dict:
    """
    Evaluate frozen features with a single linear layer (Ridge regression).

    Labels are z-score normalised before fitting.
    Reports MSE on normalised labels for α and ζ.
    """
    # Feature normalisation
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train_feats)
    X_va = scaler.transform(val_feats)

    results = {}
    for name, y_tr_raw, y_va_raw in [
        ('alpha', train_alpha, val_alpha),
        ('zeta',  train_zeta,  val_zeta),
    ]:
        mu, sigma = y_tr_raw.mean(), y_tr_raw.std() + 1e-8
        y_tr = (y_tr_raw - mu) / sigma
        y_va = (y_va_raw - mu) / sigma

        reg = Ridge(alpha=ridge_alpha)
        reg.fit(X_tr, y_tr)
        mse = mean_squared_error(y_va, reg.predict(X_va))
        results[f'lp_mse_{name}'] = mse
        print(f'  [Linear Probe] {name:5s}  MSE = {mse:.4f}')

    return results


print('linear_probe_eval defined.')


## 18. kNN Regression Evaluation


In [ ]:
def knn_eval(
    train_feats : np.ndarray, train_alpha : np.ndarray, train_zeta : np.ndarray,
    val_feats   : np.ndarray, val_alpha   : np.ndarray, val_zeta   : np.ndarray,
    k: int = 20,
) -> dict:
    """
    Evaluate frozen features with kNN regression (cosine distance).
    Reports MSE on normalised labels for α and ζ.
    """
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train_feats)
    X_va = scaler.transform(val_feats)

    results = {}
    for name, y_tr_raw, y_va_raw in [
        ('alpha', train_alpha, val_alpha),
        ('zeta',  train_zeta,  val_zeta),
    ]:
        mu, sigma = y_tr_raw.mean(), y_tr_raw.std() + 1e-8
        y_tr = (y_tr_raw - mu) / sigma
        y_va = (y_va_raw - mu) / sigma

        knn = KNeighborsRegressor(n_neighbors=k, metric='cosine', n_jobs=-1)
        knn.fit(X_tr, y_tr)
        mse = mean_squared_error(y_va, knn.predict(X_va))
        results[f'knn_mse_{name}'] = mse
        print(f'  [kNN k={k}]      {name:5s}  MSE = {mse:.4f}')

    return results


print('knn_eval defined.')


## 19. Main Training Script


In [ ]:
def main(cfg: Config = cfg):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)
    print(f'\n{"="*60}')
    print(f'V-JEPA 2  |  active_matter  |  device: {device}')
    print(f'{"="*60}\n')

    # ── Data ──────────────────────────────────────────────────────────
    train_loader, val_loader = build_dataloaders(cfg)
    steps_per_epoch = len(train_loader) // cfg.grad_accum_steps
    total_steps     = cfg.epochs * steps_per_epoch
    print(f'Steps per epoch : {steps_per_epoch}')
    print(f'Total steps     : {total_steps}\n')

    # ── Model ─────────────────────────────────────────────────────────
    model = VJEPA2(cfg).to(device)
    print('Parameter count:')
    count_parameters(model)
    print()

    # ── Optimizer (separate weight decay groups) ───────────────────────
    no_decay_names = {'bias', 'norm', 'pos_embed', 'mask_token'}
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(nd in name for nd in no_decay_names):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = optim.AdamW([
        {'params': decay_params,    'weight_decay': cfg.weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ], lr=cfg.lr, betas=(0.9, 0.999))

    scaler = torch.cuda.amp.GradScaler() if cfg.device == 'cuda' else None

    # ── Schedules ─────────────────────────────────────────────────────
    lr_schedule  = build_lr_schedule(cfg, steps_per_epoch)
    ema_schedule = build_ema_schedule(cfg, total_steps)

    # ── Resume ────────────────────────────────────────────────────────
    start_epoch = 0
    global_step = 0
    latest = find_latest_checkpoint(cfg.checkpoint_dir)
    if latest:
        print(f'Resuming from {latest}')
        start_epoch, global_step = load_checkpoint(model, optimizer, scaler, latest, cfg)
        start_epoch += 1

    # ── WandB ─────────────────────────────────────────────────────────
    if cfg.use_wandb and HAS_WANDB:
        import dataclasses
        wandb.init(
            project=cfg.wandb_project,
            config=dataclasses.asdict(cfg),
            resume='allow',
        )

    # ── Training loop ─────────────────────────────────────────────────
    best_val_loss = float('inf')

    for epoch in range(start_epoch, cfg.epochs):
        print(f'\n{"─"*60}')
        print(f'Epoch {epoch+1}/{cfg.epochs}')

        train_loss, global_step = train_one_epoch(
            model, train_loader, optimizer,
            lr_schedule, ema_schedule, global_step, epoch, cfg, scaler,
        )
        print(f'  Train Loss: {train_loss:.4f}')

        log_dict = {'epoch': epoch + 1, 'train_loss': train_loss}

        # ── Periodic evaluation ───────────────────────────────────────
        do_eval = (epoch % cfg.eval_interval == 0) or (epoch == cfg.epochs - 1)
        if do_eval:
            print('  Extracting features ...')
            tr_f, tr_a, tr_z = extract_features(model, train_loader, cfg)
            va_f, va_a, va_z = extract_features(model, val_loader,   cfg)

            print('  Linear Probe:')
            lp = linear_probe_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z)
            print('  kNN:')
            kn = knn_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z, k=cfg.knn_k)
            log_dict.update({**lp, **kn})

        if cfg.use_wandb and HAS_WANDB:
            wandb.log(log_dict)

        # ── Save checkpoint ───────────────────────────────────────────
        if epoch % cfg.save_interval == 0 or epoch == cfg.epochs - 1:
            save_checkpoint(model, optimizer, scaler, epoch, global_step, train_loss, cfg)

    print('\nTraining complete!')
    if cfg.use_wandb and HAS_WANDB:
        wandb.finish()

    return model


print('main() defined. Run the next cell to start training.')


## 20. Launch Training

> **Note:** Download the dataset first (see README).  
> Set `NETID` at the top of Cell 4 to your NYU NetID.


In [ ]:
# Uncomment to start training
# trained_model = main(cfg)


## 21. Standalone Evaluation (post-training)


In [ ]:
def evaluate_checkpoint(checkpoint_path: str, cfg: Config = cfg):
    """
    Load a saved checkpoint and run linear probe + kNN evaluation.
    Call this independently after training.
    """
    device = torch.device(cfg.device)
    model  = VJEPA2(cfg).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint: epoch {ckpt["epoch"]}, step {ckpt["global_step"]}')

    _, val_loader   = build_dataloaders(cfg)
    train_loader, _ = build_dataloaders(cfg)

    print('Extracting train features ...')
    tr_f, tr_a, tr_z = extract_features(model, train_loader, cfg)
    print('Extracting val features ...')
    va_f, va_a, va_z = extract_features(model, val_loader, cfg)

    print('\nLinear Probe Results:')
    lp = linear_probe_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z)

    print('\nkNN Results:')
    kn = knn_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z, k=cfg.knn_k)

    return {**lp, **kn}


# Usage:
# results = evaluate_checkpoint('/scratch/kk5991/checkpoints/vjepa2/vjepa2_epoch0099.pt')
print('evaluate_checkpoint() defined.')
